# 07 · The official RLM library

The paper's authors publish `rlms` (github.com/alexzhang13/rlm). Running it on the same questions checks that our
from-scratch version behaves like the reference one, and shows what a mature implementation adds.

In [1]:
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from rlm_logistics import methods
from rlm_logistics.budget import Budget
from rlm_logistics.llm import LLM, Reply
from rlm_logistics.rlm import RLM

ROOT = Path("..")
load_dotenv(ROOT / ".env")
budget = Budget()   # hard spending cap on all paid calls; the limit is BUDGET_USD in .env
DEV = ROOT / "results/dev"
meta = json.loads((ROOT / "data/processed/contexts.json").read_text())
qs = pd.read_json(ROOT / "data/processed/questions.jsonl", lines=True)
lines = (ROOT / "data/processed/context_2m.txt").read_text(encoding="utf-8").split("\n")
context = "\n".join(lines[:meta["32k"]["n_records"]])
sample = qs[qs["size"] == "32k"].groupby("tier").head(1).to_dict("records")

## Same model, same questions
`rlms` supports Anthropic directly, so it runs on the same Claude Haiku 5.5 as our version (depth 1).
`prompt` = the context (stored in its REPL), `root_prompt` = the question the root model sees.
Its calls bypass our client, so their cost is computed from the reported tokens and charged to the same spending cap.

**Windows note:** `rlms` writes the context to a temp file with the system encoding, which fails on Portuguese text.
Start Jupyter with `PYTHONUTF8=1` (UTF-8 mode), e.g. `set PYTHONUTF8=1` before `jupyter lab`.

**Compatibility patch.** `rlms` reads only the first block of a Claude reply (`content[0].text`). Claude Haiku 5.5
thinks by default, so the first block is a *thinking* block and the library crashes. The patch below keeps the library's
request and usage tracking unchanged and only joins the reply's text blocks.

In [2]:
import rlm.clients.anthropic as official_client

def _text(response):
    return "".join(b.text for b in response.content if b.type == "text")

def completion(self, prompt, model=None):
    messages, system = self._prepare_messages(prompt)
    model = model or self.model_name
    response = self.client.messages.create(model=model, max_tokens=self.max_tokens, messages=messages,
                                           **({"system": system} if system else {}))
    self._track_cost(response, model)
    return _text(response)

async def acompletion(self, prompt, model=None):
    messages, system = self._prepare_messages(prompt)
    model = model or self.model_name
    response = await self.async_client.messages.create(model=model, max_tokens=self.max_tokens, messages=messages,
                                                       **({"system": system} if system else {}))
    self._track_cost(response, model)
    return _text(response)

official_client.AnthropicClient.completion, official_client.AnthropicClient.acompletion = completion, acompletion

In [3]:
import os, sys
assert sys.flags.utf8_mode, "start Jupyter with PYTHONUTF8=1 (see the note above)"
from rlm import RLM as OfficialRLM

haiku = LLM.from_config("final")
official = OfficialRLM(backend="anthropic", backend_kwargs={"model_name": haiku.model, "api_key": os.environ["ANTHROPIC_API_KEY"]}, max_depth=1, max_iterations=12)

def ask_official(q):
    budget.check()
    res = official.completion(prompt=context, root_prompt=q["question"])
    usage = res.usage_summary.model_usage_summaries.values()
    tin, tout = sum(u.total_input_tokens for u in usage), sum(u.total_output_tokens for u in usage)
    cost = (tin * haiku.price[0] + tout * haiku.price[1]) / 1e6
    budget.charge(cost, "official rlms")
    return Reply(f"ANSWER: {res.response}", tin, tout, cost, res.execution_time)

rows = [methods.run("rlm_official", q, lambda: ask_official(q), haiku.model, DEV) for q in sample]
pd.DataFrame(rows)[["tier", "score", "input_tokens", "cost", "seconds"]]

,tier,score,input_tokens,cost,seconds
0,T1,1.000000,8589,0.001061,4.8
1,T2,0.000000,21091,0.003614,18.2
2,T3,0.583333,34503,0.008320,119.6


## Side by side with ours (depth 1)

In [4]:
ours = methods.logged("rlm_depth1", DEV)
pd.DataFrame([{"tier": q["tier"], "ours": ours.get(q["id"], {}).get("score"),
               "official": methods.logged("rlm_official", DEV)[q["id"]]["score"]} for q in sample])

,tier,ours,official
0,T1,1.000000,1.000000
1,T2,1.000000,0.000000
2,T3,0.727273,0.583333


## What the official version adds
| | Ours | `rlms` |
|---|---|---|
| Core loop (REPL, `llm_query`, FINAL) | yes | yes |
| Batched / parallel sub-calls | no (one at a time) | `llm_query_batched`, up to 4 concurrent |
| Isolated sandboxes | restricted namespace | Docker, Modal, E2B, ... |
| Compaction of long histories | no | optional |
| Trajectory viewer | JSON files | built-in visualizer |

Ours is ~200 lines and every line can be explained. The official one is what you would run in production.